In [1]:
import pandas as pd 
import numpy as np 
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score


In [2]:
ufos = pd.read_csv('./data/ufos.csv')
ufos.head()

,datetime,city,state,country,shape,duration (seconds),duration (hours/min),comments,date posted,latitude,longitude
0,10/10/1949 20:30,san marcos,tx,us,cylinder,2700.0,45 minutes,This event took place in early fall around 194...,4/27/2004,29.883056,-97.941111
1,10/10/1949 21:00,lackland afb,tx,NaN,light,7200.0,1-2 hrs,1949 Lackland AFB&#44 TX. Lights racing acros...,12/16/2005,29.384210,-98.581082
2,10/10/1955 17:00,chester (uk/england),NaN,gb,circle,20.0,20 seconds,Green/Orange circular disc over Chester&#44 En...,1/21/2008,53.200000,-2.916667
3,10/10/1956 21:00,edna,tx,us,circle,20.0,1/2 hour,My older brother and twin sister were leaving ...,1/17/2004,28.978333,-96.645833
4,10/10/1960 20:00,kaneohe,hi,us,light,900.0,15 minutes,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.418056,-157.803611


In [3]:
ufos = pd.DataFrame({'Seconds': ufos['duration (seconds)'], 'Country': ufos['country'],'Latitude': ufos['latitude'],'Longitude': ufos['longitude']})
ufos.Country.unique()

array(['us', nan, 'gb', 'ca', 'au', 'de'], dtype=object)

In [4]:
#drop all null values
#kep only important sightings between 1 and 60 sec
ufos.dropna()
ufos = ufos[(ufos['Seconds'] >= 1) & (ufos['Seconds'] <= 60)]
ufos.info()

<class 'pandas.core.frame.DataFrame'>
Index: 29360 entries, 2 to 80330
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Seconds    29360 non-null  float64
 1   Country    25863 non-null  object 
 2   Latitude   29360 non-null  float64
 3   Longitude  29360 non-null  float64
dtypes: float64(3), object(1)
memory usage: 1.1+ MB


In [5]:
#convert text values for countires to a number
from sklearn.preprocessing import LabelEncoder
ufos['Country'] = LabelEncoder().fit_transform(ufos['Country'])
ufos

,Seconds,Country,Latitude,Longitude
2,20.0,3,53.200000,-2.916667
3,20.0,4,28.978333,-96.645833
14,30.0,4,35.823889,-80.253611
18,20.0,5,32.364167,-64.678611
23,60.0,4,45.582778,-122.352222
...,...,...,...,...
80321,3.0,4,36.529722,-87.359444
80322,15.0,5,50.465843,22.891814
80323,60.0,4,29.651389,-82.325000
80326,20.0,4,34.101389,-84.519444


In [6]:

X = ufos.drop(columns='Country')
y = ufos['Country']

In [7]:
X

,Seconds,Latitude,Longitude
2,20.0,53.200000,-2.916667
3,20.0,28.978333,-96.645833
14,30.0,35.823889,-80.253611
18,20.0,32.364167,-64.678611
23,60.0,45.582778,-122.352222
...,...,...,...
80321,3.0,36.529722,-87.359444
80322,15.0,50.465843,22.891814
80323,60.0,29.651389,-82.325000
80326,20.0,34.101389,-84.519444


In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=42)

In [9]:
#training with logistics regression


In [10]:
logReg = LogisticRegression()
logReg.fit(X_train, y_train)
predictions = logReg.predict(X_test)

c:\Users\Anast\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [11]:
print(classification_report(y_test, predictions))
print('accuracy score:', accuracy_score(y_test, predictions))

              precision    recall  f1-score   support

           0       0.79      0.93      0.86        45
           1       0.00      0.00      0.00       255
           2       0.00      0.00      0.00         7
           3       0.56      0.60      0.58       146
           4       0.86      1.00      0.93      4725
           5       0.68      0.19      0.30       694

    accuracy                           0.85      5872
   macro avg       0.48      0.46      0.44      5872
weighted avg       0.80      0.85      0.80      5872

accuracy score: 0.8497956403269755


c:\Users\Anast\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Anast\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Anast\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

In [12]:
import pickle
model_file = 'ufo_model.pkl'
pickle.dump(logReg, open(model_file, 'wb'))

In [14]:
X_new = pd.DataFrame([[50, 44, -12]], columns=['Seconds','Latitude', 'Longitude' ])
print(logReg.predict(X_new))

[5]


In [15]:
model = pickle.load(open('ufo_model.pkl', 'rb'))